In [1]:
import os 
import sys
import dotenv 

from pathlib import Path
from dotenv import load_dotenv

from langchain_chroma import Chroma # this is VectorDB
from langchain_text_splitters import RecursiveCharacterTextSplitter # To Split the text


In [2]:
# Configure Environment
# Reset cwd first
ROOT_DIR = Path(os.getcwd()).parent
OUTPUT_DIR = ROOT_DIR / "output"
KNOWLEDGE_BASE = ROOT_DIR / "knowledge_base"
DATA_DIR = ROOT_DIR / "data"


In [3]:
print(f"Root of the project path: {ROOT_DIR}")
print(f"Data Path: {DATA_DIR}")
print(f"Output Path: {OUTPUT_DIR}")
print(f"Knowledge Base Path: {KNOWLEDGE_BASE}")


Root of the project path: /home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag
Data Path: /home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag/data
Output Path: /home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag/output
Knowledge Base Path: /home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag/knowledge_base


## API KEYS
---

Load API KEYS Properly for anything that is needed

In [4]:
load_dotenv()
JINA_API_KEY = os.getenv("JINA_API_KEY") # load JINA API KEY - if using Jina Embeddings Model
HF_API_KEY = os.getenv("HUGGINGFACE_API_KEY") # load HUGGINFACE API Key - if using models from hugging face

# print(f"Jina: {JINA_API_KEY}")
# print(f"Hugging Face: {HF_API_KEY}")


## Data Setup
---

- Using the data given in the assignment instructions
- Get it from Kaggle using kagglehub



In [5]:
import kagglehub
import shutil

# make sure tha data directory exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

download_path = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")

# Move files to DATA_DIR
for file in Path(download_path).iterdir():
    shutil.copy(file, DATA_DIR / file.name)

# Files must be in our DATA_DIR now
print(f"Files copied to: {DATA_DIR}")


Files copied to: /home/bhavik/Dropbox/edu/smu/winter/deep_learning/MCDA5511/a2-rag/data


## Section 1: Data Exploration
---

### 1. Data Selection & Curation
The dataset consists of **51,774** scientific paper abstracts from ArXiv. This corpus is specialized in Computer Science and Statistics, providing a dense knowledge base for RAG implementation.

### 2. Processing Documentation
Instead of sampling, we analyzed the **entire dataset** to ensure full coverage of the vocabulary. The processing involved:
- **Normalization**: Lowercasing and tokenization of summaries.
- **Term Parsing**: Evaluating the string-formatted category lists into Python objects for frequency analysis.

### 3. Summary Statistics
The statistics below describe the scale and complexity of the dataset:

| Statistic | Value |
|-----------|-------|
| Total Documents | 51,774 |
| Avg Abstract Length | 173 words |
| Vocabulary Size | ~59,600 unique words |

![Doc Length Distribution](../plots/doc_length_histogram_1772768180077.png)

### 4. Topic Frequencies
The dataset is dominated by **Computer Vision (cs.CV)** and **Machine Learning (cs.LG)**, which together appear in over 90% of the documents.

![Topic Frequencies](../plots/topic_frequency_chart_1772768284632.png)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import re
from collections import Counter

# 1. Load Data
df = pd.read_csv(DATA_DIR / "arxiv_data.csv")

# 2. Calculate Basic Statistics
df['summary_len'] = df['summaries'].apply(lambda x: len(str(x).split()))
vocab_size = len(set(re.findall(r'\w+', " ".join(df['summaries'].astype(str)).lower())))

print(f"Total Documents: {len(df)}")
print(f"Average Summary Length: {df['summary_len'].mean():.2f} words")
print(f"Vocabulary Size: {vocab_size}")

# 3. Topic Identification & Frequency
def parse_terms(x):
    try: return ast.literal_eval(x)
    except: return []

df['parsed_terms'] = df['terms'].apply(parse_terms)
all_terms = [term for sublist in df['parsed_terms'] for term in sublist]
term_counts = Counter(all_terms)

print("\nTop 10 Topics:")
for topic, count in term_counts.most_common(10):
    print(f"{topic}: {count} ({count/len(df)*100:.2f}%)")

# 4. Visualizations
plt.figure(figsize=(10, 5))
sns.histplot(df['summary_len'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of Document Lengths')
plt.show()


## Section 2: RAG Construction
---

### Model Configuration

This system follows a **Retrieval-Augmented Generation (RAG)** architecture, where a retrieval model first identifies relevant documents and a generative language model then produces an answer using that retrieved context. In this framework, the language model contributes **parametric knowledge stored in its trained weights**, while the retrieval component provides **non-parametric knowledge from the external document collection** (sourced from the **arXiv Paper Abstracts dataset**, where each document corresponds to the abstract of a scientific paper), allowing the system to generate answers grounded in the dataset. In essence, the RAG pipeline is primarily about constructing a **strong prompt** for the language model.

- **Embeddings:** `BAAI/bge-small-en`  
- **Embedding generation:** `sentence-transformers`  
- **Generative model:** `Qwen/Qwen2.5-0.5B-Instruct`

Document and query embeddings are generated using the **BAAI/bge-small-en** model through the **SentenceTransformers** library. This model produces 384-dimensional embeddings optimized for semantic search and retrieval tasks.

All document embeddings are computed once and stored locally as a NumPy array (`embeddings.npy`). The associated metadata (titles, summaries, and terms) is stored in `metadata.jsonl`. During retrieval, the user query is embedded using the same model, and cosine similarity is used to identify the most relevant documents.

The retrieved documents are then passed to the **Qwen2.5-0.5B-Instruct** language model, which generates the final answer based on a **fixed prompt template**, the **user query**, and the **retrieved document context**. The model runs locally using the HuggingFace **Transformers** library.

---

### Assignment Instructions

Construct your RAG model. Recommended components for the simplest possible implementation are
provided below, but you are free to choose whatever components you wish. If you choose your own,
write a few sentences explaining your rationale.
- Embeddings: BAAI/bge-small-en
- Embedding generation: sentence-transformers
- Retrieval: Retrieve top k most similar documents to the query using cosine similarity. Start with a
small value like k=3 and amend later if necessary. If you chose longer documents for your
dataset, you may also need to experiment with chunk size.
- Generation: Generate an answer based on a prompt, the user query, and the retrieved
documents. Informally experiment with a few different prompts to see what works best. A small
generative model that you can use is QWEN 3 0.6B, but for this component in particular, itwould be a good idea to experiment with larger models to see if they give you better results. But
again, remember that the focus of this assignment is not to build the best performing RAG.


As an alternative to the above implementation, you may use an AutoRAG tool if you prefer.

In [30]:
import os
print(os.getcwd())

c:\Users


In [31]:
import os
os.chdir(r"C:\Users\ter-k\5511-a2-rag")

In [32]:
import sys
sys.path.append("..")

In [33]:
from src.rag_pipeline import run_rag, RagConfig

In [34]:
cfg = RagConfig()

In [36]:
run_rag("What is optimization?", cfg)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 966.42it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 429.73it/s, Materializing param=model.norm.weight]                              


{'query': 'What is optimization?',
 'k': 3,
 'retrieved': [{'id': 36170,
   'title': 'Learning to Optimize',
   'summary': 'Algorithm design is a laborious process and often requires many iterations of\nideation and validation. In this paper, we explore automating algorithm design\nand present a method to learn an optimization algorithm, which we believe to be\nthe first method that can automatically discover a better algorithm. We\napproach this problem from a reinforcement learning perspective and represent\nany particular optimization algorithm as a policy. We learn an optimization\nalgorithm using guided policy search and demonstrate that the resulting\nalgorithm outperforms existing hand-engineered algorithms in terms of\nconvergence speed and/or the final objective value.',
   'terms': "['cs.LG', 'cs.AI', 'math.OC', 'stat.ML']",
   'title_len': 20,
   'summary_len': 656,
   'doc_text_len': 678,
   'score': 0.8569499850273132},
  {'id': 24231,
   'title': 'Sample Efficient Graph-B

## Section 3: Construct a Dataset for Q/A Pairs
---

Minimum 15 Questions:

Format: Questions, Retrieved documents, topic labels, Response (Answer)
Coverage: Topics present in dataset, and topics outside of dataset

---

### Assignment Instrctions

Construct a dataset of Q-A pairs by creating a set of questions covering a couple of different topics, and
for each question store the retrieved documents, their topic labels, and the generated response. The
more Q-A pairs you generate the better – 15 at a minimum. Try to find questions that demonstrate the
model working both well and poorly. One thing you might try to produce poor performance is to ask
questions about topics that are not present in the data.


In [ ]:
# Section 3 Code Cells (Can be multiple cells)


## Section 4: Manual Review of Retreived Documents
---

Write Answer Here


---
### Assignment Instrctions

Manually review the documents that were retrieved for each question. Label each retrieved document as
correct or incorrect. On a best-efforts basis, identify whether any documents should have been retrieved
that were not and label those as well. Measure precision, recall, F1-score, and accuracy and comment on
the results. Make sure you understand how to calculate these metrics for information retrieval
specifically. Do you detect any differences in performance between topics that are well-represented in
the data compared with those that are not? (This last question may not be feasible if the number of Q-A
pairs is small, in which case you can just explain that.)
In your own words, explain what cosine similarity is, how it is used for retrieval, and what its limitations
are. Can you find an instance where retrieval failed due to one or more of these limitations?

In [ ]:
# Section 4 Code Cells (Can be multiple cells)


## Section 5: Manual Review of Responses
---

Write Answer Here


---
### Assignment Instrctions

Now manually review the generated responses by comparing them against the retrieved documents. Can
you identify any instances where the model hallucinated (i.e. produced information that was not in the
retrieved documents)? Comment on the overall quality of the generated responses. (If you wish you can
write a ground truth response for each question and use cosine similarity or another metric to
quantitatively assess the generated responses. If you choose to do this, comment on the limitations of the
approach, similar as you did in question 4.)

In [ ]:
# Section 5 Code Cells (Can be multiple cells)


## Section 6: LLM as a Judge
---

Write Answer Here


---
### Assignment Instrctions

It is common practice to use an “LLM as a judge” to automate evaluation of RAG systems. Use an LLM of
your choice to automate your model testing pipeline (i.e use it to redo questions 3 – 5). This will allow you
to generate many more test cases to measure performance metrics. Inspect the results. Can you find any
instances where the LLM has assessed RAG performance incorrectly? Overall, how useful was the LLM in
assessing RAG performance?

In [ ]:
# Section 6 Code Cells (Can be multiple cells)
